# Who can get to a parkrun? London access (Post 1)
Sources (downloaded by `fetch_data.py`): parkrun events feed (`images.parkrun.com/events.json`); London Datastore borough and LSOA 2021 boundaries; ONS Census 2021 TS007A (population) and TS045 (car availability) via Nomis; TfL PTAL 2015 by LSOA 2011 (London Datastore); DfT NaPTAN stops. No parkrun results data. Distances are straight-line metres in EPSG:27700 (British National Grid), from LSOA 2021 geometric centroids.

In [1]:
import sys; sys.path.insert(0, ".")
import pandas as pd
import london_parkrun as lp
pd.set_option("display.width", 160)

boroughs = lp.load_boroughs()
events = lp.filter_london_5k(lp.load_events(), boroughs)
assert events["short_name"].is_unique
lsoa = lp.lsoa_access(lp.attach_ptal(lp.load_lsoa(), lp.load_lsoa11_ptal()), events)
stations = lp.load_stations()
print(len(events), "events;", len(lsoa), "LSOAs;", int(lsoa.population.sum()), "people")

65 events; 4994 LSOAs; 8799631 people


## Q1 How many, and where?

In [2]:
by_borough = events.sjoin(boroughs[["borough", "geometry"]], predicate="within").groupby("borough").size()
print(by_borough.sort_values(ascending=False).to_string())
print("boroughs with none:", sorted(set(boroughs.borough) - set(by_borough.index)))

borough
Greenwich                 5
Bromley                   5
Hounslow                  5
Wandsworth                4
Southwark                 4
Croydon                   4
Richmond upon Thames      3
Lewisham                  3
Haringey                  3
Havering                  3
Enfield                   2
Harrow                    2
Barnet                    2
Ealing                    2
Bexley                    2
Hammersmith and Fulham    2
Redbridge                 2
Hackney                   1
Newham                    1
Waltham Forest            1
Tower Hamlets             1
Brent                     1
Lambeth                   1
Merton                    1
Camden                    1
Kingston upon Thames      1
Islington                 1
Hillingdon                1
Barking and Dagenham      1
boroughs with none: ['City of London', 'Kensington and Chelsea', 'Sutton', 'Westminster']


## Q2 How many Londoners live near one?

In [3]:
cov = lp.coverage_by_threshold(lsoa)
print(cov.assign(share=lambda d: (100 * d.share).round(1)).to_string(index=False))
print("London median distance (m):", round(lp.weighted_median(lsoa.dist_m, lsoa.population)))

 threshold_m  population  share
        1000     1257977   14.3
        2000     4698845   53.4
        5000     8617916   97.9
London median distance (m): 1916


## Q3 Parkrun deserts

In [4]:
DESERT_M = 3000
desert = lsoa[lsoa.dist_m > DESERT_M]
print("people > 3 km:", int(desert.population.sum()), f"({100 * desert.population.sum() / lsoa.population.sum():.1f}%)")
print(desert.groupby("borough").population.sum().sort_values(ascending=False).head(10).to_string())
print(lsoa.nlargest(5, "dist_m")[["lsoa21nm", "borough", "nearest_event", "dist_m"]].to_string(index=False))

people > 3 km: 1519439 (17.3%)
borough
Hillingdon              150538
Brent                   134731
Westminster             119401
Enfield                 115652
Croydon                 110934
Kingston upon Thames    103896
Bexley                  103133
Sutton                  100181
Barking and Dagenham     95627
Redbridge                83788
       lsoa21nm    borough    nearest_event       dist_m
Hillingdon 003A Hillingdon           Harrow 10725.892829
Hillingdon 003B Hillingdon           Harrow 10153.039707
Hillingdon 003C Hillingdon           Harrow  9027.444190
Hillingdon 003E Hillingdon Stockley Country  8799.573972
   Bromley 042D    Bromley      Riddlesdown  8222.663147


## Q4 Walk from a station?

In [5]:
sa = lp.station_access(events, stations)
for col, label in (("station_m", "any rail/Tube/DLR/tram station"), ("tube_m", "a Tube station")):
    print(f"within 1 km of {label}: {(sa[col] <= 1000).sum()} of {len(sa)}")
print(sa.sort_values("station_m", ascending=False)[["short_name", "station", "station_m"]].head(8).round(0).to_string(index=False))

within 1 km of any rail/Tube/DLR/tram station: 44 of 65
within 1 km of a Tube station: 22 of 65
      short_name                         station  station_m
Wimbledon Common Southfields Underground Station     1811.0
      Crane Park            Whitton Rail Station     1681.0
         Burgess  Elephant & Castle Rail Station     1620.0
 Roundshaw Downs        Purley Oaks Rail Station     1500.0
      Sunny Hill   Colindale Underground Station     1433.0
         Bromley       Hayes (Kent) Rail Station     1423.0
   Richmond Park  Richmond (London) Rail Station     1374.0
        Kingston       Hampton Wick Rail Station     1356.0


## Q5 No car, far away (and PTAL); boroughs

In [6]:
gap = lp.car_free_gap(lsoa)
print("London no-car household share:", round(gap.no_car_households.sum() / gap.households.sum(), 3))
print("people far (>2 km):", int(gap.loc[gap.far, "population"].sum()))
print("people far AND carless:", int(gap.loc[gap.gap, "population"].sum()))
print("…and low PTAL (0–2):", int(gap.loc[gap.gap & gap.low_ptal, "population"].sum()))
print(gap[gap.gap].groupby("borough").population.sum().sort_values(ascending=False).head(8).to_string())
print(lp.borough_access(lsoa).round({"median_dist_m": 0, "share_within": 3}).to_string(index=False))

London no-car household share: 0.421
people far (>2 km): 4100786
people far AND carless: 1491938
…and low PTAL (0–2): 207381
borough
Westminster               176764
Hackney                   147469
Newham                    144374
Camden                    110660
Brent                      96648
Lambeth                    93781
Kensington and Chelsea     76353
Tower Hamlets              70441
               borough  population  median_dist_m  share_within
  Kingston upon Thames      168103         3369.0         0.148
           Westminster      204235         3236.0         0.121
        City of London        8579         3235.0         0.000
                Sutton      209640         2964.0         0.186
            Hillingdon      305923         2925.0         0.212
  Barking and Dagenham      218840         2821.0         0.276
                 Brent      339832         2662.0         0.305
                Merton      215200         2509.0         0.291
                Bexley     

## Q6 Catchments

In [7]:
cat = lp.catchments(lsoa, events)
print(cat.head(10).to_string(index=False)); print(cat.tail(5).to_string(index=False))
print("median catchment:", int(cat.population.median()))

     short_name  population  lsoas
Highbury Fields      332737    195
Wormwood Scrubs      300301    173
         Morden      288723    167
         Harrow      278704    155
 Tooting Common      237114    134
       Mile End      236145    127
      Battersea      228335    149
        Barking      226470    116
Hampstead Heath      217719    135
      Gladstone      210798    115
        short_name  population  lsoas
          Hanworth       48682     26
Foots Cray Meadows       47859     29
           Dulwich       35526     21
     Richmond Park       30950     18
     Bedfont Lakes       20415     10
median catchment: 126297


## Boundary check: parkruns just outside London
The analysis above only counts the 65 events inside Greater London. Some Londoners live nearer to an event just over the boundary, so this cell repeats the headline numbers with every UK 5k event within 10 km of London.

In [8]:
uk5k = lp.load_events().pipe(lambda e: e[(e.countrycode == lp.UK) & (e.seriesid == lp.FIVE_K)]).to_crs(lp.BNG)
nearby = uk5k[uk5k.within(boroughs.union_all().buffer(10_000))].reset_index(drop=True)
wide = lp.lsoa_access(lsoa.drop(columns=["nearest_event", "dist_m"]), nearby)
print(len(nearby), "events within 10 km;", len(nearby) - len(events), "of them outside London")
for name, d in (("London only", lsoa), ("with nearby", wide)):
    cov = lp.coverage_by_threshold(d)
    print(f"{name:12s} within 1/2/5 km: {(100 * cov.share).round(1).tolist()}  median m: {round(lp.weighted_median(d.dist_m, d.population))}  >3 km: {100 * d.loc[d.dist_m > 3000, 'population'].sum() / d.population.sum():.1f}%")
shift = (lp.borough_access(lsoa).set_index("borough").median_dist_m - lp.borough_access(wide).set_index("borough").median_dist_m).round(0)
print(shift[shift > 0].sort_values(ascending=False).to_string())
print(wide[~wide.nearest_event.isin(events.short_name)].groupby("nearest_event").population.sum().sort_values(ascending=False).head(8).to_string())

93 events within 10 km; 28 of them outside London
London only  within 1/2/5 km: [14.3, 53.4, 97.9]  median m: 1916  >3 km: 17.3%
with nearby  within 1/2/5 km: [14.3, 54.0, 99.3]  median m: 1905  >3 km: 16.0%
borough
Sutton                  570.0
Enfield                 123.0
Kingston upon Thames     39.0
Bexley                   32.0
Hillingdon               21.0
nearest_event
Nonsuch           107143
Gunpowder          63821
Dartford Heath     43159
South Oxhey        40523
Roding Valley      22821
Brentwood          16168
Banstead Woods     13433
Rickmansworth      12581
